# 04: Multi-Task MuRIL Training Pipeline (Kaggle GPU)

Fine-tunes `google/muril-base-cased` with 5 independent classification heads using PyTorch AdamW.

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
import pandas as pd
from src.data.preprocessing import MultiTaskLabelEncoder, MultitaskSafetyDataset
from src.models.multitask_model import MultitaskSafetyModel
from src.utils.config import ModelConfig, TRAINED_MODEL_DIR
from src.utils.helpers import get_device, set_seed

set_seed(42)
device = get_device()
config = ModelConfig()

# Initialize Tokenizer and Model
model_name = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = MultitaskSafetyModel(model_name=model_name, config=config).to(device)

print(f"Initialized Multi-Task Model on {device}")

In [ ]:
# Prepare DataLoaders
from src.data.loader import load_or_create_dataset
from src.data.preprocessing import group_aware_split

df = load_or_create_dataset()
train_df, val_df, test_df = group_aware_split(df)
encoder = MultiTaskLabelEncoder()

train_dataset = MultitaskSafetyDataset(
    texts=train_df['text'].tolist(),
    labels=encoder.encode_dataframe(train_df),
    tokenizer=tokenizer,
    max_length=config.max_length
)
train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
print(f"Total training batches: {len(train_loader)}")

In [ ]:
# Training Loop
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
epochs = config.epochs

model.train()
for epoch in range(epochs):
    total_epoch_loss = 0.0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = {k: v.to(device) for k, v in batch['labels'].items()}

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs['loss']
        loss.backward()
        optimizer.step()

        total_epoch_loss += loss.item()

    avg_loss = total_epoch_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{epochs} - Combined Multi-Task Loss: {avg_loss:.4f}")

# Save trained checkpoint
TRAINED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), TRAINED_MODEL_DIR / "best_multitask_model.pt")
print(f"Checkpoint saved to {TRAINED_MODEL_DIR / 'best_multitask_model.pt'}")